<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 1</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">1.1</span>Riesgo de mercado con Monte Carlo</div>
</div>

**Objetivo.** calcular VaR y Expected Shortfall (ES) de un portafolio de tres formas y compararlas.

**Lo que vamos a hacer**

- Descargar precios y calcular retornos diarios.
- Medir VaR/ES con el método histórico y el paramétrico.
- Simular 100.000 escenarios con Monte Carlo (Cholesky).
- Comparar los tres métodos.

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. Datos

In [ ]:
# EC = Ecopetrol, CIB = Bancolombia (ADR en NYSE)
tickers = ["EC", "CIB", "AAPL", "MSFT", "GLD"]
prices = download_prices(tickers, start="2019-01-01")
tickers = list(prices.columns)
weights = np.repeat(1 / len(tickers), len(tickers))      # portafolio igual-ponderado
rets = to_returns(prices, log=False)                      # retornos simples diarios
port = rets @ weights
print(f"{len(rets)} días · {len(tickers)} activos")
prices.tail(3)

## 2. VaR y ES históricos

El **VaR** al 95% es la pérdida que se supera solo el 5% de los días. El **ES** es la pérdida *promedio* en ese 5% peor.

> ### ✏️ Ejercicio 1 — Función var_es
> Devuelve `(VaR, ES)` como pérdidas positivas a partir de una serie de retornos.

In [ ]:
def var_es(x, alpha=0.95):
    x = np.asarray(x)
    # TODO: calcula el cuantil (1-alpha) y el promedio de lo que cae por debajo
    return None, None

In [ ]:
def var_es(x, alpha=0.95):
    x = np.asarray(x)
    var = -np.quantile(x, 1 - alpha)
    es = -x[x <= -var].mean()
    return var, es

In [ ]:
from scipy.stats import norm

mu, sigma = port.mean(), port.std()
z = norm.ppf(0.05)
var_param = -(mu + z * sigma)
es_param = -(mu - sigma * norm.pdf(z) / 0.05)
print("Histórico   VaR %.2f%%  ES %.2f%%" % tuple(100 * np.array(var_es(port))))
print("Paramétrico VaR %.2f%%  ES %.2f%%" % (100 * var_param, 100 * es_param))

## 3. Monte Carlo

Simulamos retornos con la misma media y covarianza. Para que los activos se muevan juntos usamos la descomposición de **Cholesky**: si $\Sigma = LL^\top$, entonces $\mu + Lz$ con $z\sim N(0,I)$ tiene covarianza $\Sigma$.

> ### ✏️ Ejercicio 2 — Simular el portafolio
> Genera `sim_port` con 100.000 retornos simulados del portafolio.

In [ ]:
import time
n_sims = 100_000
rng = np.random.default_rng(42)
mu_v, cov = rets.mean().values, rets.cov().values

t0 = time.perf_counter()
# TODO: sim_port = ...   (pista: np.linalg.cholesky y rng.standard_normal)
sim_port = None
print(f"{n_sims:,} escenarios en {time.perf_counter() - t0:.3f} s")

In [ ]:
import time
n_sims = 100_000
rng = np.random.default_rng(42)
mu_v, cov = rets.mean().values, rets.cov().values

t0 = time.perf_counter()
L = np.linalg.cholesky(cov)
z = rng.standard_normal((n_sims, len(weights)))
sim_port = (mu_v + z @ L.T) @ weights
print(f"{n_sims:,} escenarios en {time.perf_counter() - t0:.3f} s")

## 4. Comparación

In [ ]:
res = pd.DataFrame({"Histórico": var_es(port), "Paramétrico": (var_param, es_param),
                    "Monte Carlo": var_es(sim_port)}, index=["VaR 95%", "ES 95%"]).T
display((res * 100).round(2).astype(str) + " %")

v, e = var_es(sim_port)
fig, ax = plt.subplots()
ax.hist(sim_port, bins=120, color="#d9d9d9")
ax.axvline(-v, color="#A6192E", ls="--", label=f"VaR 95% = {v:.2%}")
ax.axvline(-e, color="#1f1f1f", ls="--", label=f"ES 95% = {e:.2%}")
ax.set_title("Distribución simulada del retorno diario"); ax.legend(); plt.show()

> ### ✏️ Ejercicio 3 — Más extremo y más largo
> Calcula VaR y ES al **99%** y escala el VaR a **10 días** con la regla de la raíz del tiempo.

In [ ]:
# TODO

In [ ]:
v99, e99 = var_es(sim_port, 0.99)
print(f"VaR 99% 1d: {v99:.2%} | ES 99% 1d: {e99:.2%}")
print(f"VaR 99% 10d (regla √t): {v99 * np.sqrt(10):.2%}")

## Para llevarse

- El ES siempre es mayor que el VaR: mide *qué tan mala* es la cola, no solo dónde empieza.
- Monte Carlo permite cualquier distribución; aquí la normal multivariante reproduce el método paramétrico.
- 100.000 escenarios tardan una fracción de segundo; en Excel serían 100.000 filas recalculadas.

---
*Excel para mirar, Python para repetir.*